### Pre-modeling

In [2]:
import pandas as pd
import time

from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, classification_report, confusion_matrix

In [3]:
df = pd.read_csv('../data/processed/Telco-Customer-Churn-Processed.csv')

In [4]:
X = df.drop(columns=['Churn'])
y = df['Churn']

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=86, stratify=y
)

---

We have to choose which metric to use.

In our business problem:
- if model makes type 1 error -> company will trigger its standard retention protocols (discounts, sending gifts etc.) basically unnecessary costs

- if model makes type 2 error -> customer will leave, which will decrease revenue of the company, it also leads to damage to reputation and to spend more money on marketing

In [6]:
df['Churn'].value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

We see that dataset is imbalanced, so accuracy can't be used there.

Precision is also not the best option because cost of false positive is not that bad.

Cost of false negative on the other side is too high, so the best option is `recall` or `f1`, but i would stick up to recall, since FN is more dangerous and `threshold` in our case would be `0.3`

---

### Model selection (RF vs LGBM vs XGB)

In [7]:
# Robust to class imbalance 
ratio = (y_train == 0).sum() / (y_train == 1).sum()
THRESHOLD = 0.3

#### Random Forest 

In [8]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    class_weight='balanced',
    random_state=75,
    n_jobs=-1
)

rf.fit(X_train, y_train)

proba = rf.predict_proba(X_test)[:, 1]
y_pred = (proba >= THRESHOLD).astype(int)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.50      0.66      1035
           1       0.41      0.95      0.57       374

    accuracy                           0.62      1409
   macro avg       0.69      0.73      0.62      1409
weighted avg       0.82      0.62      0.64      1409



Based on classification report we see, that model found 95% of churn customers

But on the other side it predicts only 41% of them correctly -> Model's making a lot of type 1 error    

---

#### LGBM

In [10]:
from lightgbm import LGBMClassifier

In [22]:
lgbm = LGBMClassifier(
    n_estimators=300,
    max_depth=6,
    class_weight='balanced',
    random_state=75,
    n_jobs=-1
)

start = time.perf_counter()

lgbm.fit(X_train, y_train)

proba = lgbm.predict_proba(X_test)[:, 1]
y_pred = (proba >= THRESHOLD).astype(int)

end = time.perf_counter()

print(f"Training took {end - start:.3f} seconds")

print(classification_report(y_test, y_pred))

Training took 1.567 seconds
              precision    recall  f1-score   support

           0       0.92      0.70      0.79      1035
           1       0.50      0.83      0.62       374

    accuracy                           0.73      1409
   macro avg       0.71      0.76      0.71      1409
weighted avg       0.81      0.73      0.75      1409



Model caught 83% of actual churn customers (13% worse than baseline) -> which is overall not that bad, but model makes a lot of false positive mistakes (only 50% of predicted churns are actually churns) 

---

#### XGBoost

In [13]:
from xgboost import XGBClassifier

In [21]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    class_weight='balanced',
    random_state=75,
    n_jobs=-1
)

start = time.perf_counter()

xgb.fit(X_train, y_train)

proba = xgb.predict_proba(X_test)[:, 1]
y_pred = (proba >= THRESHOLD).astype(int)

end = time.perf_counter()

print(f"Training took {end - start:.3f} seconds")
print(classification_report(y_test, y_pred))

Training took 1.130 seconds
              precision    recall  f1-score   support

           0       0.87      0.82      0.85      1035
           1       0.58      0.67      0.62       374

    accuracy                           0.78      1409
   macro avg       0.73      0.75      0.73      1409
weighted avg       0.79      0.78      0.79      1409



XGBoost has slightly better accuracy, precision and it's 38% faster than LGBM

But on the recall metric it managed to find only 67% of all churns (16% worse)

---

I'm gonna choose LGBM since it managed to find more churn customers which for company is more significant since FN cost is very high, but not FP

Yes, i know that i'm gonna sacrifice speed and overall accuracy, but what this'll cost to company? Additional 10$ discount offer instead of customer leaving

In real world i'd decide what to sacrifice with company but since it's pet project i'm choosing `LGBM`

---

### Threshold testing

In [33]:
from sklearn.metrics import precision_score, recall_score, f1_score

proba = lgbm.predict_proba(X_test)[:, 1]

for t in [0.2, 0.25, 0.3, 0.35, 0.4]:
    preds = (proba >= t).astype(int)
    prec = precision_score(y_test, preds, pos_label=1)
    rec = recall_score(y_test, preds, pos_label=1)
    f1 = f1_score(y_test, preds, pos_label=1)
    print(f"Threshold: {t} has precision: {prec:.3f}, recall: {rec:.3f}, f1: {f1:.3f}")

Threshold: 0.2 has precision: 0.449, recall: 0.885, f1: 0.595
Threshold: 0.25 has precision: 0.473, recall: 0.853, f1: 0.609
Threshold: 0.3 has precision: 0.498, recall: 0.826, f1: 0.622
Threshold: 0.35 has precision: 0.516, recall: 0.799, f1: 0.627
Threshold: 0.4 has precision: 0.538, recall: 0.783, f1: 0.638


Best precision-recall tradeoff found on 0.25 threshold, so i'll use it

In [34]:
THRESHOLD = 0.25

---

### Hyperparameters tuning

In [36]:
import optuna
from lightgbm import LGBMClassifier
from sklearn.metrics import recall_score
from sklearn.model_selection import train_test_split

In [ ]:
def objective(trial):
    params = {
        "n_estimators":trial.suggest_int("n_estimators", 300, 800),
        "learning_rate":trial.suggest_float("learning_rate", 0.05, 0.2, log=True),
        "max_depth":trial.suggest_int("max_depth", 4, 10),
        "num_leaves":trial.suggest_int("num_leaves", 24, 60),
        "subsample":trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_split_gain":trial.suggest_float("min_split_gain", ),

        "reg_lambda":trial.suggest_float("reg_lambda", 0, 5, log=True),
        "reg_alpha":trial.suggest_float("reg_alpha", 0, 5, log=True),

        "scale_pos-weight":(y_train == 0).sum() / (y_train == 1).sum(),
        "random_state":87,
        "n_jobs":-1,
        "eval_metric":"logloss",
        "class_weight":"scale_pos_weight"
    }